# PedRL — Pedagogical RL on GSM8K (Colab)

Implementation of [Pedagogical RL](https://noahziems.com/pedagogical-rl): train a **privileged teacher** (it sees the gold answer) with GRPO to maximize `correctness × G_spike`, where `G_spike` is a spike-aware learnability score under the **frozen student** — then distill the teacher into the student with **surprisal-gated** cross-entropy.

**Runtime:** use a GPU runtime (`Runtime → Change runtime type → T4/L4/A100`).

| preset | what | time on T4 | time on A100 |
|---|---|---|---|
| `smoke` | pipeline check (16 problems, 6 GRPO steps) | ~10–15 min | ~4 min |
| `poc` | real run (256 problems, 120 GRPO steps) | ~2–3 h | ~40 min |


In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
%pip install -q -U "trl>=0.17.0" "peft>=0.14.0" "datasets>=2.19.0" "accelerate>=0.34.0" "transformers>=4.49.0"


## 2. Get the code

**Option A (recommended):** push the `PedRL` folder to a GitHub repo and set `REPO_URL` below.

**Option B:** leave `REPO_URL` empty and upload a zip of the repo (`zip -r pedrl.zip PedRL`) when prompted.

In [ ]:
import os, zipfile

REPO_URL = ""  # e.g. "https://github.com/<you>/PedRL.git"

if REPO_URL:
    !git clone {REPO_URL} PedRL
else:
    from google.colab import files
    print("Upload pedrl.zip (a zip of the PedRL folder)...")
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall(".")

# locate the repo root (folder containing run.py)
root = None
for cand in ["PedRL", "pedrl-main", "."]:
    if os.path.exists(os.path.join(cand, "run.py")):
        root = cand
        break
assert root, "could not find run.py — check the upload/clone"
%cd {root}
!ls

## 3. Smoke test (~10 min on T4)

Runs the full pipeline on a tiny slice: baseline eval → teacher GRPO → corpus → gated assimilation → student eval. Numbers will be noisy — this only verifies everything runs.

In [ ]:
!python run.py all --preset smoke

## 4. Proof of concept

The real run. Each stage runs in its own process, so you can also re-run stages individually (e.g. after a Colab disconnect — completed stages leave artifacts in `outputs/`).

Watch the `[reward] acc=... G=... r_ped=...` lines during the teacher stage: accuracy should stay high (the teacher knows the answer) while `G` — learnability under the frozen student — climbs.

In [ ]:
# fresh outputs for the PoC (the smoke test wrote to outputs/ too)
!rm -rf outputs
!python run.py all --preset poc

## 5. Results

In [ ]:
import json, glob

for path in sorted(glob.glob("outputs/eval_*.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['tag']:>22}: pass@1 = {r['accuracy']:.3f}  (n={r['n']})")

In [ ]:
# peek at a teacher demonstration the student found most learnable
import json
rows = [json.loads(l) for l in open("outputs/distill_corpus.jsonl")]
rows.sort(key=lambda r: -r["g_spike"])
r = rows[0]
print(f"G_spike={r['g_spike']:.3f}  gold={r['answer']}\n")
print(r["completion_text"][:1500])

## 6. Optional: ablation & stage 3

- **No-gating ablation** — is the surprisal gate doing work, or is this just rejection-sampling SFT?
- **Stage 3** — continue with *standard* GRPO on the assimilated student (correctness-only reward, no privileged info).

In [ ]:
# ablation: plain SFT on the same corpus (writes eval_student_sft_ablation.json)
!python run.py assimilate --preset poc --no-gating
!python run.py eval-student --preset poc --no-gating

In [ ]:
# optional stage 3: standard GRPO on the student (writes eval_student_rl.json)
!python run.py student-rl --preset poc

---
References: [Pedagogical RL blog](https://noahziems.com/pedagogical-rl) · [OPSD](https://github.com/siyan-zhao/OPSD) · [GRPO](https://arxiv.org/abs/2402.03300)